# Week 1: Grounding with SAP Generative AI Hub

Welcome to week 1 of the April Developer Challenge on AI! This week you'll learn how to implement Retrieval-Augmented Generation (RAG) using SAP's Generative AI Hub orchestration service with document grounding.

## What You'll Build

In this exercise, you'll create a RAG workflow that grounds an LLM in information about German social services, specifically:
- The German NGO "die Tafel" (food bank organization)
- German basic income support ("Grundsicherung")

## Key Concepts

The orchestration service of Generative AI Hub lets you use all the available models with the same codebase. A deployment of the orchestration service comes automatically with the SAP AI Core instance and you can access all available models simply by changing the model name parameter. You can also use grounding, prompt templating, data masking and content filtering capabilities.

By the end of this exercise, you'll understand how to:
- Connect to the orchestration service
- Configure document grounding with a vector database
- Build a complete RAG workflow
- Stream responses with grounded context

Let's get started!

### Install the Cloud SDK for AI

In [ ]:
%pip install sap-ai-sdk-gen

In [ ]:
"""You do not need this for this Code Challenge. Only run when run locally with .env file, NOT in trial environment with config.json file from provided environment."""
# from dotenv import load_dotenv
# load_dotenv()

# AICORE_ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/D276250e074fc028"

### Import the packages you want to use

In [ ]:
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.config import GroundingModule, OrchestrationConfig
from gen_ai_hub.orchestration.models.document_grounding import DocumentGrounding, DocumentGroundingFilter, DataRepositoryType, GroundingFilterSearch
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.service import OrchestrationService

### Data Repository ID
The data repository id is the id of your grounding pipeline that you can create with the grounding service of Generative AI Hub. More information on the [Grounding - SAP Help Website](https://help.sap.com/docs/sap-ai-core/generative-ai/grounding). The ID below is the pre-created pipeline containing the information about:
- The German NGO "die Tafel" (food bank organization)
- German basic income support ("Grundsicherung")

In [ ]:
DATA_REPOSITORY_ID = "d7834fe4-83f0-4d6e-8583-1be833a2ca0b"

### Configure the LLM and the prompt
You can also try out different models here! Check out all the available models here: [3437766 - Availability of Generative AI Models](https://me.sap.com/notes/3437766)

In [ ]:
llm = LLM(
    name="gemini-2.5-flash",
    parameters={
        'temperature': 0.0,
    }
)
template = Template(
            messages=[
                SystemMessage("You are a helpful assistant for the German citizen center."),
                UserMessage("""Answer the request by providing relevant answers that fit to the request.
                Request: {{ ?user_query }}
                Context:{{ ?grounding_response }}
                """),
            ]
        )

### Create an orchestration configuration that specifies the grounding capability
Now you are adding the grounding pipeline to your orchestration workflow. Try out different values for `max_chunk_count`. Changing the amount of data retrieved from the grounding service.

In [ ]:
# Set up Document Grounding
filters = [
            DocumentGroundingFilter(
                id="vector", data_repositories=[DATA_REPOSITORY_ID],
                search_config=GroundingFilterSearch(max_chunk_count=5),
                data_repository_type=DataRepositoryType.VECTOR.value
            )
        ]

grounding_config = GroundingModule(
            type="document_grounding_service",
            config=DocumentGrounding(input_params=["user_query"], output_param="grounding_response", filters=filters)
        )

config = OrchestrationConfig(
    template=template,
    llm=llm,
    grounding=grounding_config
)

### Stream the response

In [ ]:
orchestration_service = OrchestrationService(
#    api_url=AICORE_ORCHESTRATION_DEPLOYMENT_URL,
    config=config
)

response = orchestration_service.stream(
    template_values=[
        TemplateValue(
            name="user_query",
            #TODO Here you can change the user prompt into whatever you want to ask the model
            value="What is die Tafel in Germany?"
        )
    ]
)

for chunk in response:
    print(chunk.orchestration_result.choices[0].delta.content)

Please submit a screenshot of the LLM's response directly under the [Week 1 Discussion]()

### Want to do more?
Check out the [documentation](https://help.sap.com/doc/generative-ai-hub-sdk/CLOUD/en-US/_reference/orchestration-service2.html) and add Data Masking and Content Filtering to your workflow!

[Week 1 explained in more detail](week1_grounding_README.md)

[SUBMIT YOUR RESPONSE HERE](https://community.sap.com/t5/artificial-intelligence-forum/ai-developer-challenge-week-1-grounding-with-sap-generative-ai-hub/td-p/14366146)